# 第9章　浮动利率债券

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch09_floating_rate.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch09_floating_rate.ipynb)

复现例9.1（DM=QM 平价）、例9.2（DM 与价格）、图9-1（浮息 vs 固息）、折现利差与利差久期、QuantLib 对拍。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import frn, plotting
from fi.cashflow import make_cashflows
from fi.pricing import price_bond
plotting.use_chinese_style()


## 例9.1　DM = QM 时浮息债平价（与基准水平无关）


In [ ]:
for L in (0.01, 0.02, 0.03, 0.04):
    p = frn.price_frn(reference=L, quoted_margin=0.005, disc_margin=0.005, n_periods=8, freq=4)
    print(f'基准={L:.0%}: 价格={p:.6f}')


## 例9.2　DM 偏离 QM -> 折价；由价格反求 DM


In [ ]:
for dm in (0.005, 0.008, 0.012):
    print(f'DM={dm*100:.1f}%: 价格={frn.price_frn(0.02, 0.005, dm, 8, 4):.4f}')
dm_solved = frn.discount_margin(99.50, reference=0.02, quoted_margin=0.005, n_periods=8, freq=4)
print(f'\n价格 99.50 -> 折现利差 DM = {dm_solved*100:.4f}%')


## 图9-1　浮息债 vs 固息债：利率变动下的价格稳定性（编程实验 7）


In [ ]:
refs = np.linspace(0.01, 0.04, 61)
frn_p = [frn.price_frn(L, 0.005, 0.005, 8, 4) for L in refs]
fix_p = [price_bond(*make_cashflows(0.025, 2, freq=4, face=100), L+0.005, freq=4) for L in refs]
fig, ax = plotting.new_axes()
ax.plot(refs*100, frn_p, label='浮息债（DM=QM）')
ax.plot(refs*100, fix_p, label='固息债（票息 2.5%）')
ax.axhline(100, ls=':', color='gray')
ax.set_xlabel('市场利率 (%)'); ax.set_ylabel('价格')
ax.set_title('图9-1　浮息债 vs 固息债的价格稳定性'); ax.legend()
fig.tight_layout()


### 利差久期：浮息债对 DM 的敏感度（编程实验 8）

浮息债利率久期很低，但利差久期与到期期限相当。


In [ ]:
def spread_duration(ref, qm, dm, n, freq, face=100, bp=1e-4):
    p0 = frn.price_frn(ref, qm, dm, n, freq, face)
    pu = frn.price_frn(ref, qm, dm+bp, n, freq, face)
    pd = frn.price_frn(ref, qm, dm-bp, n, freq, face)
    return (pd - pu) / (2 * p0 * bp)

sd = spread_duration(0.02, 0.005, 0.005, 8, 4)
print(f'2年期季付浮息债 利差久期 ≈ {sd:.4f} 年（与到期期限相当）')
# 对比：同期限固息债的修正久期
from fi import risk
cf, t = make_cashflows(0.025, 2, freq=4, face=100)
print(f'同期限固息债 修正久期 ≈ {risk.modified_duration(cf, t, 0.025, 4):.4f} 年')


## 9.6　QuantLib `FloatingRateBond` 对拍


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026)
ql.Settings.instance().evaluationDate = today
fwd_h = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.02, ql.Actual365Fixed()))
idx = ql.IborIndex('MyIbor', ql.Period(3, ql.Months), 0, ql.CNYCurrency(),
                   ql.NullCalendar(), ql.Unadjusted, False, ql.Actual365Fixed(), fwd_h)
sched = ql.Schedule(today, today + ql.Period(2, ql.Years), ql.Period(3, ql.Months),
                    ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
bond = ql.FloatingRateBond(0, 100.0, sched, idx, ql.Actual365Fixed(), ql.Unadjusted, 0,
                           [1.0], [0.005], [], [], False, 100.0, today)
disc = ql.FlatForward(today, 0.025, ql.Actual365Fixed())   # 基准 + DM = 2% + 0.5%
bond.setPricingEngine(ql.DiscountingBondEngine(ql.YieldTermStructureHandle(disc)))
print(f'fi.frn      价格 = {frn.price_frn(0.02, 0.005, 0.005, 8, 4):.4f}')
print(f'QuantLib    价格 = {bond.cleanPrice():.4f}  (DM=QM 时≈面值)')


---

> 小结：`fi.frn` 透明展示『DM=QM→重定价日平价、基准变动不影响价格』，与 QuantLib `FloatingRateBond` 一致；
> 浮息债利率久期低、利差久期高——抗加息但承担利差风险。
